Pixeltable's Nebius Token Factory integration enables you to access Nebius language and embedding models via an OpenAI-compatible API.

### Prerequisites

- A Nebius Token Factory account with an API key (https://tokenfactory.nebius.com/)

### Important notes

- Nebius usage may incur costs based on your Nebius plan.
- Be mindful of sensitive data and consider security measures when integrating with external services.

First you'll need to install the required libraries and enter a Nebius API key. Nebius uses the OpenAI SDK as its Python API, so we need to install it in addition to Pixeltable.

In [ ]:
%pip install -qU pixeltable openai

In [2]:
import getpass
import os

if 'NEBIUS_API_KEY' not in os.environ:
    os.environ['NEBIUS_API_KEY'] = getpass.getpass('Nebius API Key:')

Now let's create a Pixeltable directory to hold the tables for our demo.

In [ ]:
import pixeltable as pxt

# Remove the 'nebius_demo' directory and its contents, if it exists
pxt.drop_dir('nebius_demo', force=True)
pxt.create_dir('nebius_demo')

## Chat completions

Create a Table: In Pixeltable, create a table with columns to represent your input data and the columns where you want to store the results from Nebius.

In [4]:
from pixeltable.functions import nebius

chat_t = pxt.create_table('nebius_demo/chat', {'input': pxt.String})

messages = [{'role': 'user', 'content': chat_t.input}]

chat_t.add_computed_column(
    output=nebius.chat_completions(
        messages=messages,
        model='meta-llama/Llama-3.3-70B-Instruct',
        model_kwargs={
            # Optional dict with parameters for the Nebius API
            'max_tokens': 300,
            'temperature': 0.7,
        },
    )
)
chat_t.add_computed_column(
    response=chat_t.output.choices[0].message.content
)

Created table 'chat'.
Added 0 column values with 0 errors in 0.01 s
Added 0 column values with 0 errors in 0.00 s


No rows affected.

In [5]:
# Start a conversation
chat_t.insert(
    [
        {'input': 'What is the capital of France?'},
        {'input': 'Explain quantum computing in one sentence.'},
    ]
)
chat_t.select(chat_t.input, chat_t.response).head()

Inserted 2 rows with 0 errors in 4.43 s (0.45 rows/s)


input,response
What is the capital of France?,The capital of France is Paris.
Explain quantum computing in one sentence.,"Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data at an exponentially faster rate than classical computers, by utilizing the unique properties of quantum bits or qubits that can exist in multiple states simultaneously."


## Embeddings

Nebius currently serves the embedding model `Qwen/Qwen3-Embedding-8B`. By default it produces 4096-dimensional embeddings, which exceed Pixeltable's embedding-index limit of 4000 dimensions. Request a smaller size via `model_kwargs` when you need an indexable embedding.

In [6]:
emb_t = pxt.create_table('nebius_demo/embeddings', {'input': pxt.String})
emb_t.add_computed_column(
    embedding=nebius.embeddings(
        input=emb_t.input, model='Qwen/Qwen3-Embedding-8B'
    )
)

Created table 'embeddings'.
Added 0 column values with 0 errors in 0.00 s


No rows affected.

In [7]:
emb_t.insert(
    [
        {
            'input': 'Nebius Token Factory provides language and embedding models.'
        }
    ]
)

Inserted 1 row with 0 errors in 8.15 s (0.12 rows/s)


1 row inserted.

In [8]:
emb_t.head()

input,embedding
Nebius Token Factory provides language and embedding models.,[ 0.001 0.004 -0.008 -0.036 0.012 -0.021 ... -0.003 0.004 0.014 -0.004 0.008 -0.015]


To build an embedding index, truncate to an indexable size (for example 1024) with `model_kwargs`:

In [9]:
indexed = nebius.embeddings.using(
    model='Qwen/Qwen3-Embedding-8B', model_kwargs={'dimensions': 1024}
)
emb_t.add_embedding_index('input', embedding=indexed)

In [10]:
emb_t.insert([{'input': 'Another sentence for you to index.'}])

Inserted 1 row with 0 errors in 3.62 s (0.28 rows/s)


1 row inserted.

In [11]:
sim = emb_t.input.similarity(string='Indexing sentences is fun.')
emb_t.select(emb_t.input, sim=sim).order_by(sim, asc=False).collect()

input,sim
Another sentence for you to index.,0.811
Nebius Token Factory provides language and embedding models.,0.573


### Learn more

To learn more about advanced techniques like RAG operations in Pixeltable, check out the [RAG Operations in Pixeltable](https://docs.pixeltable.com/howto/use-cases/rag-operations) tutorial.

If you have any questions, don't hesitate to reach out.